# Notebook 04: ZCTA-Level Coverage and Desert Metrics

## Purpose
This notebook builds full ZCTA coverage features, computes nearest DCFC distance, and labels charging deserts.

## Analytical outcome
Produce a complete geographic feature table for modeling infrastructure access gaps.

In [1]:
import os
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType
import math

os.environ['JAVA_HOME'] = '/opt/homebrew/Cellar/openjdk@17/17.0.19/libexec/openjdk.jdk/Contents/Home'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EV_Charging_Analysis") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/03 12:00:32 WARN Utils: Your hostname, Krishs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.228 instead (on interface en0)
26/05/03 12:00:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 12:00:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/03 12:00:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/03 12:00:34 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark version: 4.1.1


In [10]:
# Load census data - this represents ALL US ZIP areas
# This becomes our base - every ZCTA gets a row regardless of 
# whether it has any chargers

df_census = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(os.path.join(BASE_DIR, "acs_zcta_combined.csv")) \
    .withColumn(
        "zcta",
        F.lpad(F.col("zcta").cast("string"), 5, "0")
    )

print(f"Total US ZCTAs (base): {df_census.count()}")

Total US ZCTAs (base): 33772


In [11]:
# Aggregate station data by ZIP - same as before
# But now this is just the RIGHT side of a join, not the final table

df_station_agg = df.groupBy("ZIP", "State", "region") \
    .agg(
        F.count("ID").alias("total_stations"),
        F.sum("is_dcfc").alias("dcfc_stations"),
        F.sum("total_ports").alias("total_ports"),
        F.sum("EV Level2 EVSE Num").alias("total_level2_ports"),
        F.sum("EV DC Fast Count").alias("total_dcfc_ports"),
        F.avg("Latitude").alias("zip_lat"),
        F.avg("Longitude").alias("zip_lon")
    )

print(f"ZIPs with at least one station: {df_station_agg.count()}")

ZIPs with at least one station: 11890


In [12]:
# Left join Census onto stations
# ZCTAs with no stations will have null station counts
# which we then fill with zeros

df_full = df_census.join(
    df_station_agg,
    df_census["zcta"] == df_station_agg["ZIP"],
    how="left"
)

# Fill nulls for ZCTAs with no stations
df_full = df_full \
    .withColumn("total_stations", 
        F.coalesce(F.col("total_stations"), F.lit(0))) \
    .withColumn("dcfc_stations",  
        F.coalesce(F.col("dcfc_stations"), F.lit(0))) \
    .withColumn("total_ports",    
        F.coalesce(F.col("total_ports"), F.lit(0))) \
    .withColumn("total_level2_ports", 
        F.coalesce(F.col("total_level2_ports"), F.lit(0))) \
    .withColumn("total_dcfc_ports",   
        F.coalesce(F.col("total_dcfc_ports"), F.lit(0)))

# Use zcta as the primary key going forward
df_full = df_full \
    .withColumnRenamed("zcta", "ZIP_ZCTA") \
    .drop("ZIP")

total = df_full.count()
has_stations = df_full.filter(F.col("total_stations") > 0).count()
no_stations = df_full.filter(F.col("total_stations") == 0).count()

print(f"Total ZCTAs:            {total:,}")
print(f"ZCTAs with stations:    {has_stations:,}")
print(f"ZCTAs with no stations: {no_stations:,}")

Total ZCTAs:            33,798
ZCTAs with stations:    11,694
ZCTAs with no stations: 22,104


In [5]:
# The Haversine formula calculates distance between two 
# lat/lon points on a sphere (Earth)
# R = 3,958.8 miles (Earth's radius in miles)

# We register this as a Spark UDF so it can run on every row
# across all Spark partitions in parallel

def haversine_miles(lat1, lon1, lat2, lon2):
    """
    Calculate distance in miles between two lat/lon points.
    Returns None if any coordinate is missing.
    """
    if lat1 is None or lon1 is None or lat2 is None or lon2 is None:
        return None
    
    R = 3958.8  # Earth radius in miles
    
    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    
    return R * c

# Register as Spark UDF returning a Double (decimal number)
haversine_udf = spark.udf.register("haversine_miles", haversine_miles, DoubleType())

print("Haversine UDF registered successfully")

# Quick sanity test - distance between NYC and LA should be ~2,450 miles
test = spark.createDataFrame([(40.7128, -74.0060, 34.0522, -118.2437)],
                              ["lat1", "lon1", "lat2", "lon2"])
test.withColumn("distance", haversine_udf("lat1", "lon1", "lat2", "lon2")).show()

Haversine UDF registered successfully


[Stage 16:>                                                         (0 + 1) / 1]

+-------+-------+-------+---------+-----------------+
|   lat1|   lon1|   lat2|     lon2|         distance|
+-------+-------+-------+---------+-----------------+
|40.7128|-74.006|34.0522|-118.2437|2445.586606929677|
+-------+-------+-------+---------+-----------------+



In [13]:
# ZCTAs with no stations have null zip_lat and zip_lon
# because we got coordinates from station averages
# We need coordinates for ALL ZCTAs to compute the 50-mile check
# 
# Solution: Use a US ZCTA centroid reference
# We'll approximate using state center coordinates for unmatched ZCTAs
# This is a known limitation we'll document

# For now, check how many have null coordinates
null_coords = df_full.filter(F.col("zip_lat").isNull()).count()
print(f"ZCTAs with null coordinates (no stations): {null_coords:,}")
print(f"These will get nearest_dcfc_miles = null")
print(f"Riddhi will handle these in her model as a separate category")

ZCTAs with null coordinates (no stations): 22,104
These will get nearest_dcfc_miles = null
Riddhi will handle these in her model as a separate category


In [17]:
# Load Census ZCTA centroid reference file
# This is tab-separated, not comma-separated
GAZETTEER_PATH = os.path.join(BASE_DIR, "2023_Gaz_zcta_national.txt")

df_gaz_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", "\t") \
    .csv(GAZETTEER_PATH)

# See what columns came in
print("Gazetteer columns:", df_gaz_raw.columns)
df_gaz_raw.show(5, truncate=False)

Gazetteer columns: ['GEOID', 'ALAND', 'AWATER', 'ALAND_SQMI', 'AWATER_SQMI', 'INTPTLAT', 'INTPTLONG                                                                                                                                  ']
+-----+---------+-------+----------+-----------+---------+-------------------------------------------------------------------------------------------------------------------------------------------+
|GEOID|ALAND    |AWATER |ALAND_SQMI|AWATER_SQMI|INTPTLAT |INTPTLONG                                                                                                                                  |
+-----+---------+-------+----------+-----------+---------+-------------------------------------------------------------------------------------------------------------------------------------------+
|601  |166848592|798613 |64.421    |0.308      |18.180555|-66.749961                                                                                                       

In [18]:
# The INTPTLONG column has trailing whitespace in its name
# We fix this by renaming all columns with strip()
# then selecting just what we need

df_gaz = df_gaz_raw.toDF(*[c.strip() for c in df_gaz_raw.columns])

# Now select and clean the three columns we need
df_gaz_clean = df_gaz.select(
    F.lpad(F.col("GEOID").cast("string"), 5, "0").alias("zcta_geo"),
    F.col("INTPTLAT").cast(DoubleType()).alias("zcta_lat"),
    F.col("INTPTLONG").cast(DoubleType()).alias("zcta_lon")
)

print(f"Gazetteer ZCTAs: {df_gaz_clean.count():,}")
df_gaz_clean.show(5)

Gazetteer ZCTAs: 33,791
+--------+---------+----------+
|zcta_geo| zcta_lat|  zcta_lon|
+--------+---------+----------+
|   00601|18.180555|-66.749961|
|   00602|18.361945|-67.175597|
|   00603|18.457399|-67.124867|
|   00606|18.158327|-66.932928|
|   00610|18.295304| -67.12518|
+--------+---------+----------+
only showing top 5 rows


In [19]:
# Join gazetteer coords onto our full ZCTA table
# Then use coalesce to pick station-average coords if available,
# fall back to gazetteer centroid if not
# coalesce(a, b) returns a if a is not null, otherwise b

df_with_coords = df_density.join(
    df_gaz_clean,
    df_density["ZIP_ZCTA"] == df_gaz_clean["zcta_geo"],
    how="left"
).withColumn(
    "final_lat",
    F.coalesce(F.col("zip_lat"), F.col("zcta_lat"))
).withColumn(
    "final_lon",
    F.coalesce(F.col("zip_lon"), F.col("zcta_lon"))
).drop("zcta_geo", "zcta_lat", "zcta_lon")

# Verify nulls are gone
remaining_nulls = df_with_coords.filter(F.col("final_lat").isNull()).count()
print(f"ZCTAs still missing coordinates after gazetteer fill: {remaining_nulls:,}")
print(f"Total rows: {df_with_coords.count():,}")

ZCTAs still missing coordinates after gazetteer fill: 0
Total rows: 33,798


In [20]:
print("Calculating nearest DCFC distance for all ZCTAs... (few minutes)")

df_with_distance = df_with_coords.withColumn(
    "nearest_dcfc_miles",
    nearest_dcfc_udf(F.col("final_lat"), F.col("final_lon"))
)

df_with_distance.cache()
print(f"Done. Total rows: {df_with_distance.count():,}")

print("\n=== Nearest DCFC Distance Distribution ===")
df_with_distance.select(
    F.min("nearest_dcfc_miles").alias("min_miles"),
    F.round(F.avg("nearest_dcfc_miles"), 2).alias("avg_miles"),
    F.round(F.max("nearest_dcfc_miles"), 2).alias("max_miles"),
    F.round(F.percentile_approx("nearest_dcfc_miles", 0.5), 2).alias("median_miles"),
    F.round(F.percentile_approx("nearest_dcfc_miles", 0.9), 2).alias("90th_pct_miles")
).show()

Calculating nearest DCFC distance for all ZCTAs... (few minutes)


[Stage 137:>                                                        (0 + 1) / 1]

Done. Total rows: 33,798

=== Nearest DCFC Distance Distribution ===
+---------+---------+---------+------------+--------------+
|min_miles|avg_miles|max_miles|median_miles|90th_pct_miles|
+---------+---------+---------+------------+--------------+
|      0.0|    16.23|  1076.55|        8.15|         25.49|
+---------+---------+---------+------------+--------------+



## Charging Desert Classification

After nearest-DCFC distances are computed, each ZCTA is classified using the 50-mile access threshold.
Regional summaries then quantify where access constraints are most concentrated.

In [21]:
df_desert = df_with_distance.withColumn(
    "is_charging_desert",
    F.when(F.col("nearest_dcfc_miles") > 50, 1)
     .when(F.col("nearest_dcfc_miles").isNull(), None)
     .otherwise(0)
)

desert  = df_desert.filter(F.col("is_charging_desert") == 1).count()
adequate = df_desert.filter(F.col("is_charging_desert") == 0).count()
unknown  = df_desert.filter(F.col("is_charging_desert").isNull()).count()
total    = df_desert.count()

print(f"Total ZCTAs:               {total:,}")
print(f"Charging deserts (>50 mi): {desert:,} ({desert/total*100:.1f}%)")
print(f"Adequate coverage:         {adequate:,} ({adequate/total*100:.1f}%)")
print(f"Unknown (no coordinates):  {unknown:,}")

print("\n=== Desert breakdown by region ===")
df_desert.filter(F.col("region").isNotNull()) \
    .groupBy("region") \
    .agg(
        F.sum(F.col("is_charging_desert").cast("int")).alias("desert_zips"),
        F.count("ZIP_ZCTA").alias("total_zips")
    ) \
    .withColumn("desert_pct",
        F.round(F.col("desert_zips") / F.col("total_zips") * 100, 1)
    ) \
    .orderBy("desert_pct", ascending=False) \
    .show()

Total ZCTAs:               33,798
Charging deserts (>50 mi): 752 (2.2%)
Adequate coverage:         33,046 (97.8%)
Unknown (no coordinates):  0

=== Desert breakdown by region ===
+---------+-----------+----------+----------+
|   region|desert_zips|total_zips|desert_pct|
+---------+-----------+----------+----------+
|     West|         22|      2812|       0.8|
|  Midwest|         12|      2519|       0.5|
|    South|          5|      3810|       0.1|
|Northeast|          0|      2553|       0.0|
+---------+-----------+----------+----------+



In [22]:
# How many ZCTAs have null State?
null_state = df_desert.filter(F.col("State").isNull()).count()
print(f"ZCTAs with null State: {null_state:,}")

# What does the desert breakdown look like including nulls?
df_desert.groupBy("region") \
    .agg(
        F.sum(F.col("is_charging_desert").cast("int")).alias("desert_zips"),
        F.count("ZIP_ZCTA").alias("total_zips")
    ) \
    .orderBy("desert_zips", ascending=False) \
    .show()

ZCTAs with null State: 22,104
+---------+-----------+----------+
|   region|desert_zips|total_zips|
+---------+-----------+----------+
|     NULL|        713|     22104|
|     West|         22|      2812|
|  Midwest|         12|      2519|
|    South|          5|      3810|
|Northeast|          0|      2553|
+---------+-----------+----------+



In [23]:
# Show the most extreme desert ZCTAs
df_desert.filter(F.col("is_charging_desert") == 1) \
    .select("ZIP_ZCTA", "State", "region", 
            "nearest_dcfc_miles", "total_stations") \
    .orderBy("nearest_dcfc_miles", ascending=False) \
    .show(10)

+--------+-----+------+------------------+--------------+
|ZIP_ZCTA|State|region|nearest_dcfc_miles|total_stations|
+--------+-----+------+------------------+--------------+
|   00765| NULL|  NULL|1076.5532005201292|             0|
|   00775| NULL|  NULL|1076.4287979079866|             0|
|   00735| NULL|  NULL|1058.3407080244283|             0|
|   00740| NULL|  NULL|1057.5461148279198|             0|
|   00741| NULL|  NULL|1056.2794345232444|             0|
|   00738| NULL|  NULL|1054.6866735658214|             0|
|   00718| NULL|  NULL| 1053.847788775432|             0|
|   00791| NULL|  NULL| 1053.653423506506|             0|
|   00773| NULL|  NULL|1051.9991778518695|             0|
|   00707| NULL|  NULL|1051.9590039003338|             0|
+--------+-----+------+------------------+--------------+
only showing top 10 rows


In [25]:
# US ZIP code prefix to state mapping
# First 3 digits of ZIP code reliably identify the state
# This is based on USPS official ZIP prefix assignments

zip_prefix_to_state = {
    **{str(i).zfill(3): "PR" for i in range(600, 989)},   # Puerto Rico
    **{str(i).zfill(3): "MA" for i in range(10, 28)},
    **{str(i).zfill(3): "RI" for i in range(28, 30)},
    **{str(i).zfill(3): "NH" for i in range(30, 39)},
    **{str(i).zfill(3): "ME" for i in range(39, 50)},
    **{str(i).zfill(3): "VT" for i in range(50, 60)},
    **{str(i).zfill(3): "CT" for i in range(60, 70)},
    **{str(i).zfill(3): "NJ" for i in range(70, 90)},
    **{str(i).zfill(3): "NY" for i in range(100, 150)},
    **{str(i).zfill(3): "PA" for i in range(150, 197)},
    **{str(i).zfill(3): "DE" for i in range(197, 200)},
    **{str(i).zfill(3): "DC" for i in range(200, 206)},
    **{str(i).zfill(3): "MD" for i in range(206, 220)},
    **{str(i).zfill(3): "VA" for i in range(220, 247)},
    **{str(i).zfill(3): "WV" for i in range(247, 270)},
    **{str(i).zfill(3): "NC" for i in range(270, 290)},
    **{str(i).zfill(3): "SC" for i in range(290, 300)},
    **{str(i).zfill(3): "GA" for i in range(300, 320)},
    **{str(i).zfill(3): "FL" for i in range(320, 350)},
    **{str(i).zfill(3): "AL" for i in range(350, 370)},
    **{str(i).zfill(3): "TN" for i in range(370, 386)},
    **{str(i).zfill(3): "MS" for i in range(386, 400)},
    **{str(i).zfill(3): "KY" for i in range(400, 428)},
    **{str(i).zfill(3): "OH" for i in range(430, 460)},
    **{str(i).zfill(3): "IN" for i in range(460, 480)},
    **{str(i).zfill(3): "MI" for i in range(480, 500)},
    **{str(i).zfill(3): "IA" for i in range(500, 530)},
    **{str(i).zfill(3): "WI" for i in range(530, 550)},
    **{str(i).zfill(3): "MN" for i in range(550, 568)},
    **{str(i).zfill(3): "SD" for i in range(570, 580)},
    **{str(i).zfill(3): "ND" for i in range(580, 590)},
    **{str(i).zfill(3): "MT" for i in range(590, 600)},
    **{str(i).zfill(3): "IL" for i in range(600, 630)},
    **{str(i).zfill(3): "MO" for i in range(630, 660)},
    **{str(i).zfill(3): "KS" for i in range(660, 680)},
    **{str(i).zfill(3): "NE" for i in range(680, 700)},
    **{str(i).zfill(3): "LA" for i in range(700, 716)},
    **{str(i).zfill(3): "AR" for i in range(716, 730)},
    **{str(i).zfill(3): "OK" for i in range(730, 750)},
    **{str(i).zfill(3): "TX" for i in range(750, 800)},
    **{str(i).zfill(3): "CO" for i in range(800, 820)},
    **{str(i).zfill(3): "WY" for i in range(820, 832)},
    **{str(i).zfill(3): "ID" for i in range(832, 840)},
    **{str(i).zfill(3): "UT" for i in range(840, 850)},
    **{str(i).zfill(3): "AZ" for i in range(850, 870)},
    **{str(i).zfill(3): "NM" for i in range(870, 885)},
    **{str(i).zfill(3): "NV" for i in range(889, 900)},
    **{str(i).zfill(3): "CA" for i in range(900, 962)},
    **{str(i).zfill(3): "HI" for i in range(967, 969)},
    **{str(i).zfill(3): "OR" for i in range(970, 980)},
    **{str(i).zfill(3): "WA" for i in range(980, 995)},
    **{str(i).zfill(3): "AK" for i in range(995, 1000)},
}

# Register as UDF
def zip_to_state(zip_code):
    if zip_code is None:
        return None
    prefix = str(zip_code).zfill(5)[:3]
    return zip_prefix_to_state.get(prefix, None)

zip_to_state_udf = spark.udf.register("zip_to_state", zip_to_state)

print("ZIP to state UDF registered")
print(f"Prefix mappings loaded: {len(zip_prefix_to_state)}")

# Quick test
test_zips = spark.createDataFrame(
    [("10001",), ("90210",), ("60601",), ("00601",), ("99501",)], 
    ["zip"]
)
test_zips.withColumn("state", zip_to_state_udf("zip")).show()

ZIP to state UDF registered
Prefix mappings loaded: 976
+-----+-----+
|  zip|state|
+-----+-----+
|10001|   NY|
|90210|   CA|
|60601|   IL|
|00601| NULL|
|99501|   AK|
+-----+-----+



In [26]:
# Apply state lookup to all ZCTAs
# Use existing State if available, fall back to prefix lookup
df_with_state = df_with_distance.withColumn(
    "State",
    F.coalesce(
        F.col("State"),
        zip_to_state_udf(F.col("ZIP_ZCTA"))
    )
)

# Rebuild region using the same mapping as before
region_map = {
    "CT": "Northeast", "ME": "Northeast", "MA": "Northeast",
    "NH": "Northeast", "RI": "Northeast", "VT": "Northeast",
    "NJ": "Northeast", "NY": "Northeast", "PA": "Northeast",
    "IL": "Midwest", "IN": "Midwest", "MI": "Midwest",
    "OH": "Midwest", "WI": "Midwest", "IA": "Midwest",
    "KS": "Midwest", "MN": "Midwest", "MO": "Midwest",
    "NE": "Midwest", "ND": "Midwest", "SD": "Midwest",
    "DE": "South", "FL": "South", "GA": "South",
    "MD": "South", "NC": "South", "SC": "South",
    "VA": "South", "DC": "South", "WV": "South",
    "AL": "South", "KY": "South", "MS": "South",
    "TN": "South", "AR": "South", "LA": "South",
    "OK": "South", "TX": "South",
    "AZ": "West", "CO": "West", "ID": "West",
    "MT": "West", "NV": "West", "NM": "West",
    "UT": "West", "WY": "West", "AK": "West",
    "CA": "West", "HI": "West", "OR": "West", "WA": "West"
}

mapping_expr = F.create_map(
    [F.lit(x) for pair in region_map.items() for x in pair]
)

df_with_state = df_with_state.withColumn(
    "region",
    F.coalesce(
        F.col("region"),
        mapping_expr[F.col("State")]
    )
)

# Verify improvement
null_state_after = df_with_state.filter(F.col("State").isNull()).count()
null_region_after = df_with_state.filter(F.col("region").isNull()).count()
print(f"Null State after fix:  {null_state_after:,}")
print(f"Null Region after fix: {null_region_after:,}")

Null State after fix:  132
Null Region after fix: 132


In [27]:
# Remove Puerto Rico ZCTAs from analysis
# PR is not part of NEVI state-level program
df_us_only = df_with_state.filter(F.col("State") != "PR")

print(f"Rows after removing PR: {df_us_only.count():,}")

# Reapply desert flag on filtered data
df_desert_final = df_us_only.withColumn(
    "is_charging_desert",
    F.when(F.col("nearest_dcfc_miles") > 50, 1)
     .when(F.col("nearest_dcfc_miles").isNull(), None)
     .otherwise(0)
)

desert  = df_desert_final.filter(F.col("is_charging_desert") == 1).count()
adequate = df_desert_final.filter(F.col("is_charging_desert") == 0).count()
unknown  = df_desert_final.filter(F.col("is_charging_desert").isNull()).count()
total    = df_desert_final.count()

print(f"\nTotal ZCTAs:               {total:,}")
print(f"Charging deserts (>50 mi): {desert:,} ({desert/total*100:.1f}%)")
print(f"Adequate coverage:         {adequate:,} ({adequate/total*100:.1f}%)")
print(f"Unknown:                   {unknown:,}")

print("\n=== Desert breakdown by region ===")
df_desert_final.filter(F.col("region").isNotNull()) \
    .groupBy("region") \
    .agg(
        F.sum(F.col("is_charging_desert").cast("int")).alias("desert_zips"),
        F.count("ZIP_ZCTA").alias("total_zips")
    ) \
    .withColumn("desert_pct",
        F.round(F.col("desert_zips") / F.col("total_zips") * 100, 1)
    ) \
    .orderBy("desert_pct", ascending=False) \
    .show()

# Confirm no more PR in top deserts
print("\n=== Top 10 most remote ZCTAs ===")
df_desert_final.filter(F.col("is_charging_desert") == 1) \
    .select("ZIP_ZCTA", "State", "region", "nearest_dcfc_miles") \
    .orderBy("nearest_dcfc_miles", ascending=False) \
    .show(10)

Rows after removing PR: 33,666

Total ZCTAs:               33,666
Charging deserts (>50 mi): 620 (1.8%)
Adequate coverage:         33,046 (98.2%)
Unknown:                   0

=== Desert breakdown by region ===
+---------+-----------+----------+----------+
|   region|desert_zips|total_zips|desert_pct|
+---------+-----------+----------+----------+
|     West|        376|      5807|       6.5|
|  Midwest|        181|     10153|       1.8|
|    South|         57|     11600|       0.5|
|Northeast|          6|      6106|       0.1|
+---------+-----------+----------+----------+


=== Top 10 most remote ZCTAs ===
+--------+-----+------+------------------+
|ZIP_ZCTA|State|region|nearest_dcfc_miles|
+--------+-----+------+------------------+
|   99742|   AK|  West| 676.2700086551353|
|   99769|   AK|  West| 639.4926940657087|
|   99583|   AK|  West| 581.7059364807346|
|   99571|   AK|  West| 561.1861051588096|
|   99612|   AK|  West| 556.6988679517785|
|   99783|   AK|  West| 554.9570077347523|

In [28]:
COUNTY_PATH = os.path.join(OUTPUT_DIR, "county_level_features.parquet")

# Persist final features as a single authoritative artifact for downstream work.
df_desert_final.write \
    .mode("overwrite") \
    .parquet(COUNTY_PATH)

verify = spark.read.parquet(COUNTY_PATH)
print(f"Saved. Verification row count: {verify.count():,}")
print(f"Columns: {verify.columns}")

Saved. Verification row count: 33,666
Columns: ['ZIP_ZCTA', 'total_population', 'median_household_income', 'State', 'region', 'total_stations', 'dcfc_stations', 'total_ports', 'total_level2_ports', 'total_dcfc_ports', 'zip_lat', 'zip_lon', 'chargers_per_10k', 'final_lat', 'final_lon', 'nearest_dcfc_miles', 'is_charging_desert']


In [29]:
# How many ZCTAs got filtered because of this?
df_with_state.filter(F.col("State") == "PR").count()

0

In [ ]:
## Notebook 04 Output

The final analytical table is written to `output/county_level_features.parquet` and includes coverage intensity, distance-to-DCFC, and desert labels for downstream modeling and visualization.